In [9]:
!pip install azure-ai-projects azure-core

In [10]:
pip install --upgrade openai azure-ai-projects azure-identity

In [11]:
from openai import OpenAI
import os
from google.colab import userdata

# Retrieve the secret key from Google Colab
api_key = userdata.get("azure-key")

client = OpenAI(
    api_key=api_key,  # Picks up the secret key from Colab environment
    base_url="https://rg-aai-demo-resource.openai.azure.com/openai/v1"
)


In [12]:
# 1. Define the input data
ai_research_data = """
Title: AI-Based Healthcare Diagnosis System

Abstract:
This research paper explores the use of Artificial Intelligence in early disease detection.
Machine learning models are trained on patient datasets to predict diseases such as diabetes and cancer.

Introduction:
Artificial Intelligence is transforming healthcare by enabling faster and more accurate diagnosis.
This study focuses on predictive modeling techniques.

Methodology:
We used supervised learning algorithms including Random Forest and Neural Networks.
The dataset was preprocessed and split into training and testing sets.

Results:
The model achieved an accuracy of 92% on the test dataset.
Neural Networks performed better compared to traditional models.

Conclusion:
AI can significantly improve early diagnosis and reduce human error in healthcare systems.
"""

# 2. Extract structured research data
try:
    response = client.responses.create(
        model="gpt-4.1",
        input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",
                        "text": """Extract the following details from the research paper and return ONLY valid JSON:

                        - Title
                        - Abstract (short summary)
                        - Methodology
                        - Key Results
                        - Conclusion
                        """
                    },
                    {
                        "type": "input_text",
                        "text": f"RESEARCH PAPER:\n{ai_research_data}"
                    }
                ]
            }
        ]
    )

    print("--- EXTRACTED RESEARCH DATA ---")
    print(response.output[0].content)

except Exception as e:
    print(f"Extraction Error: {e}")

--- EXTRACTED RESEARCH DATA ---
[ResponseOutputText(annotations=[], text='{\n  "Title": "AI-Based Healthcare Diagnosis System",\n  "Abstract": "This research paper explores the use of Artificial Intelligence in early disease detection. Machine learning models are trained on patient datasets to predict diseases such as diabetes and cancer.",\n  "Methodology": "We used supervised learning algorithms including Random Forest and Neural Networks. The dataset was preprocessed and split into training and testing sets.",\n  "Key Results": "The model achieved an accuracy of 92% on the test dataset. Neural Networks performed better compared to traditional models.",\n  "Conclusion": "AI can significantly improve early diagnosis and reduce human error in healthcare systems."\n}', type='output_text', logprobs=[])]


In [13]:
import json

try:
    # 1. Get the first item in the content list
    # 2. Access the text returned from model response
    output_text_item = response.output[0].content[0]

    # Supports both dict and object formats
    if isinstance(output_text_item, dict):
        raw_json_string = output_text_item.get("text", "")
    else:
        raw_json_string = output_text_item.text

    # Parse JSON string
    extracted_data = json.loads(raw_json_string)

    print("--- SUCCESS: RESEARCH PAPER SUMMARY PARSED ---")
    print(json.dumps(extracted_data, indent=2))

except Exception as e:
    print(f"Parsing Error: {e}")

    # Troubleshooting
    try:
        print(f"Actual content type: {type(response.output[0].content[0])}")
    except:
        print("Unable to inspect response format.")

--- SUCCESS: RESEARCH PAPER SUMMARY PARSED ---
{
  "Title": "AI-Based Healthcare Diagnosis System",
  "Abstract": "This research paper explores the use of Artificial Intelligence in early disease detection. Machine learning models are trained on patient datasets to predict diseases such as diabetes and cancer.",
  "Methodology": "We used supervised learning algorithms including Random Forest and Neural Networks. The dataset was preprocessed and split into training and testing sets.",
  "Key Results": "The model achieved an accuracy of 92% on the test dataset. Neural Networks performed better compared to traditional models.",
  "Conclusion": "AI can significantly improve early diagnosis and reduce human error in healthcare systems."
}


In [14]:
from azure.storage.blob import BlobServiceClient

# ✅ Correct connection string
AZURE_STORAGE_CONNECTION_STRING = "DefaultEndpointsProtocol=https;AccountName=scmdobuildblob;AccountKey=67eOvlgX0V2v1MnLNOdPTJL0IhSLldg3HF3VV9ISZyo61L50eRGF7z/7tlsqpgvfQBKqM2hm09rc+ASt4xJhOQ==;EndpointSuffix=core.windows.net"

blob_service_client = BlobServiceClient.from_connection_string(
    AZURE_STORAGE_CONNECTION_STRING
)

container_name = "blobstoragecontainer"

blob_client = blob_service_client.get_blob_client(
    container=container_name,
    blob="result.txt"
)

blob_client.upload_blob("Test upload", overwrite=True)

print("✅ Upload successful")

✅ Upload successful


In [16]:
import datetime

file_name = f"research_summary_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

In [17]:
blob_client = blob_service_client.get_blob_client(
    container="blobstoragecontainer",
    blob=file_name
)

blob_client.upload_blob(response.output_text, overwrite=True)

{'etag': '"0x8DE9F77C3B613A0"',
 'last_modified': datetime.datetime(2026, 4, 21, 7, 29, 46, tzinfo=datetime.timezone.utc),
 'content_md5': bytearray(b'C)b\xdd\xf7w\x10udnG\xda8\xfb\x99\xec'),
 'client_request_id': 'df6ebb90-3d53-11f1-a6b2-0242ac1c000c',
 'request_id': 'a6a3f3c2-801e-006e-5760-d1ee3b000000',
 'version': '2026-02-06',
 'version_id': None,
 'date': datetime.datetime(2026, 4, 21, 7, 29, 46, tzinfo=datetime.timezone.utc),
 'request_server_encrypted': True,
 'encryption_key_sha256': None,
 'encryption_scope': None,
 'structured_body': None}

In [18]:
print(f"✅ Stored as: {file_name}")

✅ Stored as: research_summary_20260421_072917.txt
